# Standardization

In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
pd.set_option('display.max_colwidth', None)

## Leitura do arquivo

In [3]:
# Leitura

PATH = '/home/tatiane/repositorios/brazil-health-data-project/data_pipeline'
df = pd.read_csv(f"{PATH}/data/raw/ibge_populacao/tabela6579.csv", sep=";", skiprows=3)

In [4]:
print(len(df))
df.head()

5774


,Cód.,Unidade da Federação e Município,2018,2019,2020,2021
0,11,Rondônia,1757589,1777225,1796460,1815278
1,12,Acre,869265,881935,894470,906876
2,13,Amazonas,4080611,4144597,4207714,4269995
3,14,Roraima,576568,605761,631181,652713
4,15,Pará,8513497,8602865,8690745,8777124


- Remorção de comentários na variável de código

In [5]:
# 1. Lista com os códigos de UF
codigos_uf_validos = ['11', '12', '13', '14', '15', '16', '17', '21', '22', '23', '24', '25', '26', '27', '28', '29', '31', '32', 
                      '33', '35', '41', '42', '43', '50', '51', '52', '53']

# 2. Garantia de que a coluna seja tratada como string para a análise
codigos_como_string = df['Cód.'].astype(str).str.strip()

# 3. Definindo o que é um código de MUNICIPIO válido (6 ou 7 dígitos numéricos)
municipio_valido = (codigos_como_string.str.isnumeric()) & (codigos_como_string.str.len().isin([6, 7]))

# 4. Definindo o que é um código de UF válido
uf_valida = codigos_como_string.isin(codigos_uf_validos)

# 5. O dado é considerado "Geral Válido" se for um município válido OU uma UF válida
dados_validos = municipio_valido | uf_valida

# 6. Usando o til (~) para pegar a NEGAÇÃO (tudo que NÃO é válido)
erros_df = df.loc[~dados_validos]

# 7. Resultados
print(f"Quantidade de erros após remover as UFs: {len(erros_df)}")
print(erros_df['Cód.'].unique().tolist())

Quantidade de erros após remover as UFs: 176
['Fonte: IBGE - Estimativas de População', 'Notas', 'Em 2003:\nO contingente populacional estimado para o Município de Caraíbas, no Estado da Bahia foi reduzido em 5.006 habitantes, devido ao processo de sindicância recentemente encerrado que apurou sobre enumeração ocorrida quando da realização do Censo Demográfico 2000.\n\nEm 2004:', '1 - O contingente populacional estimado para o Município de Caraíbas, no Estado da Bahia foi reduzido em 5.006 habitantes, devido ao processo de sindicância recentemente encerrado que apurou sobre enumeração ocorrida quando da realização do Censo Demográfico 2000.', '2 - Inclusive 4 municípios com instalação prevista para 1º de janeiro de 2005: Aroeiras do Itaim (PI), Figueirão (MS), Ipiranga do Norte (MT) e Itanhangá (MT).\n\nEm 2008:\nInclusive o Município de Nazária - PI, com previsão de instalação em 1º de janeiro de 2009.\n\nEm 2009:\nPor determinação judicial e para efeito de distribuição do Fundo de Pa

In [6]:
# Criando variável para guardar quem é VÁLIDO
dados_validos = municipio_valido | uf_valida

# Substituição: Onde os dados NÃO forem válidos (~dados_validos), mude o valor da coluna 'cod.' para np.nan
df.loc[~dados_validos, 'Cód.'] = np.nan

In [7]:
# Validação
erros_df = df.loc[~dados_validos]

print(f"Quantidade de erros após remover as UFs: {len(erros_df)}")
print(erros_df['Cód.'].unique().tolist())

Quantidade de erros após remover as UFs: 176
[nan]


In [8]:
# 1. Criando uma cópia temporária da coluna limpa de espaços em branco e como string
nomes_como_string = df['Unidade da Federação e Município'].astype(str).str.strip()

# 2. Filtrando tudo que for MAIOR que 40 caracteres
filtro_textos_longos = nomes_como_string.str.len() > 40

# 3. Isolando os erros para investigação
comentarios_df = df.loc[filtro_textos_longos]

# 4. Exibe a quantidade e os textos únicos encontrados (sem truncar)
import pandas as pd
pd.set_option('display.max_colwidth', None)

print(f"Quantidade de linhas com comentários longos: {len(comentarios_df)}")
print("\nComentários encontrados:")
print(comentarios_df['Unidade da Federação e Município'].unique().tolist())

Quantidade de linhas com comentários longos: 6

Comentários encontrados:
['Zero absoluto, não resultante de um cálculo ou arredondamento.\nEx: Em determinado município não existem pessoas de 14 anos de idade sem instrução.', 'Zero resultante de um cálculo ou arredondamento.\nEx: A inflação do feijão em determinada Região Metropolitana foi 0.\nDeterminado município produziu 400 kg de sementes de girassol e os dados da tabela são expressos em toneladas.', 'Valor inibido para não identificar o informante.\nEx: Determinado município só possui uma empresa produtora de cimento, logo o valor de sua produção deve ser inibido.', 'Valor não se aplica.\nEx: Não se pode obter o total da produção agrícola em determinado município quando os produtos agrícolas são contabilizados com unidades de medida distintas.', 'Valor não disponível.\nEx: A produção de feijão em determinado município não foi pesquisada ou determinado município não existia no ano da pesquisa.', 'Significa uma faixa de valores. Vari

In [9]:
# Substitui por NaN apenas na coluna 'Unidade da Federação e Município' onde o comprimento do texto passar de 40 caracteres
df.loc[filtro_textos_longos, 'Unidade da Federação e Município'] = np.nan

### First changes

In [10]:
df["uf"] = df["Unidade da Federação e Município"].str.extract(r"\((\w{2})\)")
df["municipio"] = df["Unidade da Federação e Município"].str.replace(r"\s*\(\w{2}\)", "", regex=True)
df["cod_uf"] = df["Cód."].astype(str).str[:2]

- First validation

In [11]:
df.head(100)

,Cód.,Unidade da Federação e Município,2018,2019,2020,2021,uf,municipio,cod_uf
0,11,Rondônia,1757589,1777225,1796460,1815278,NaN,Rondônia,11
1,12,Acre,869265,881935,894470,906876,NaN,Acre,12
2,13,Amazonas,4080611,4144597,4207714,4269995,NaN,Amazonas,13
3,14,Roraima,576568,605761,631181,652713,NaN,Roraima,14
4,15,Pará,8513497,8602865,8690745,8777124,NaN,Pará,15
...,...,...,...,...,...,...,...,...,...
95,1200435,Santa Rosa do Purus (AC),6362,6540,6717,6893,AC,Santa Rosa do Purus,12
96,1200450,Senador Guiomard (AC),22810,23024,23236,23446,AC,Senador Guiomard,12
97,1200500,Sena Madureira (AC),45177,45848,46511,47168,AC,Sena Madureira,12
98,1200609,Tarauacá (AC),41976,42567,43151,43730,AC,Tarauacá,12


In [12]:
cols_anos = [col for col in df.columns if col.isdigit()]

df_long = df.melt(
    id_vars=["Cód.", "municipio", "uf", "cod_uf"],
    value_vars=cols_anos,
    var_name="ano",
    value_name="populacao"
)

- Second validation

In [13]:
print(len(df_long))
df_long.head(100)

23096


,Cód.,municipio,uf,cod_uf,ano,populacao
0,11,Rondônia,NaN,11,2018,1757589
1,12,Acre,NaN,12,2018,869265
2,13,Amazonas,NaN,13,2018,4080611
3,14,Roraima,NaN,14,2018,576568
4,15,Pará,NaN,15,2018,8513497
...,...,...,...,...,...,...
95,1200435,Santa Rosa do Purus,AC,12,2018,6362
96,1200450,Senador Guiomard,AC,12,2018,22810
97,1200500,Sena Madureira,AC,12,2018,45177
98,1200609,Tarauacá,AC,12,2018,41976


In [14]:
# garantindo que está como string (evita problemas)
df_long['Cód.'] = df_long['Cód.'].astype(str)

# criando nova coluna de UF
df_long['cod_uf_novo'] = df_long['Cód.'].apply(lambda x: x if len(x) <= 2 else x[:2])

# criando coluna de código do município
df_long['cod_municipio'] = df_long['Cód.'].apply(lambda x: x if len(x) > 2 else None)

- Third validation

In [15]:
print(len(df_long))
df_long.head(100)

23096


,Cód.,municipio,uf,cod_uf,ano,populacao,cod_uf_novo,cod_municipio
0,11,Rondônia,NaN,11,2018,1757589,11,None
1,12,Acre,NaN,12,2018,869265,12,None
2,13,Amazonas,NaN,13,2018,4080611,13,None
3,14,Roraima,NaN,14,2018,576568,14,None
4,15,Pará,NaN,15,2018,8513497,15,None
...,...,...,...,...,...,...,...,...
95,1200435,Santa Rosa do Purus,AC,12,2018,6362,12,1200435
96,1200450,Senador Guiomard,AC,12,2018,22810,12,1200450
97,1200500,Sena Madureira,AC,12,2018,45177,12,1200500
98,1200609,Tarauacá,AC,12,2018,41976,12,1200609


In [16]:
df_long = df_long.drop(columns=['Cód.', 'cod_uf_novo'])

In [17]:
cols_int = [
    "populacao",
    "ano",
    "cod_uf",
    "cod_municipio"
]

for col in cols_int:

    df_long[col] = (
        pd.to_numeric(
            df_long[col],
            errors="coerce"
        )
        .astype("Int64")
    )

In [19]:
print(df_long.dtypes)

municipio        object
uf               object
cod_uf            Int64
ano               Int64
populacao         Int64
cod_municipio     Int64
dtype: object


In [20]:
# Existem linhas onde apenas a coluna 'ano' está preenchida, então excluiremos as mesmas.
df_long = df_long[
    ~(
        df_long.drop(columns=["ano"]).isnull().all(axis=1)
    )
]

- Final validation

In [21]:
print(len(df_long))
df_long.head(100)

22396


,municipio,uf,cod_uf,ano,populacao,cod_municipio
0,Rondônia,NaN,11,2018,1757589,<NA>
1,Acre,NaN,12,2018,869265,<NA>
2,Amazonas,NaN,13,2018,4080611,<NA>
3,Roraima,NaN,14,2018,576568,<NA>
4,Pará,NaN,15,2018,8513497,<NA>
...,...,...,...,...,...,...
95,Santa Rosa do Purus,AC,12,2018,6362,1200435
96,Senador Guiomard,AC,12,2018,22810,1200450
97,Sena Madureira,AC,12,2018,45177,1200500
98,Tarauacá,AC,12,2018,41976,1200609


### Writing the dataset

In [22]:
df_long.to_parquet('/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/processed/ibge_populacao/ibge_populacao2.parquet', index=False)